# Geocoding and Spatial Integration of NC Public Records (Wake & Pitt Counties)

Course final project, CSCI 6905 Geospatial Thinking in Python (East Carolina University, 2025). See the README for the pipeline, data sources and data-quality findings.

**Outputs are cleared on purpose**: the source voter file contains names and addresses, so no record-level results are published.

In [ ]:
# Setup: point BASE at the folder holding the downloaded public data (see README)
import os
from pathlib import Path
BASE = Path(os.environ.get('NC_GEO_DATA', 'data'))
(BASE / 'output').mkdir(parents=True, exist_ok=True)


## Cell 1: Imports & Paths

In [ ]:
import pandas as pd
import geopandas as gpd
import os

# Set base directory
base = str(BASE)
voter_path = os.path.join(base, "ncvoter74Pitt", "ncvoter74.txt")
addr_gdb = os.path.join(base, "PITT-addresses-06-11-2025", "PITT.gdb")
parcel_gdb = os.path.join(base, "pitt-parcels-07-11-2025", "PITT.gdb")
output_dir = os.path.join(base, "output")

os.makedirs(output_dir, exist_ok=True)


## Cell 2: Load & Clean Voter Data

In [ ]:
voters = pd.read_csv(voter_path, sep="\t", dtype=str, encoding="latin1", low_memory=False)
print("Loaded:", len(voters))

voters.columns = voters.columns.str.upper()
voters = voters[voters['VOTER_STATUS_DESC'] == 'ACTIVE']
voters = voters.dropna(subset=['RES_STREET_ADDRESS', 'RES_CITY_DESC', 'STATE_CD', 'ZIP_CODE'])

voters['FULL_ADDRESS'] = (
    voters['RES_STREET_ADDRESS'].str.strip() + ', ' +
    voters['RES_CITY_DESC'].str.strip() + ', ' +
    voters['STATE_CD'].str.strip() + ' ' +
    voters['ZIP_CODE'].str.strip()
)

print("After cleaning:", len(voters))
voters[['FULL_ADDRESS', 'PARTY_CD']].head()


## Cell 3: Load Address GDB & Geocode

In [ ]:
import os
import re
import geopandas as gpd

# === Load Address Points ===
addr = gpd.read_file(addr_gdb, layer="PITT")
print("Loaded address points:", len(addr))
addr.columns = addr.columns.str.upper()

# Improved Normalization Function
def normalize_address(s):
    if pd.isna(s): return ""
    s = s.lower()
    s = re.sub(r'#\s*\w+', '', s)                            # remove apt/unit info
    s = re.sub(r'\bdr\b', 'drive', s)
    s = re.sub(r'\brd\b', 'road', s)
    s = re.sub(r'\bst\b', 'street', s)
    s = re.sub(r'\bave\b', 'avenue', s)
    s = re.sub(r'\bblvd\b', 'boulevard', s)
    s = re.sub(r'\bln\b', 'lane', s)
    s = re.sub(r'[^\w\s]', '', s)                            # remove punctuation
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'nc\s*\d{5}$', '', s)                        # remove trailing NC and zip
    return s

# Create normalized address field
voters['ADDR_NORM'] = voters['FULL_ADDRESS'].apply(normalize_address)
addr['ADDR_NORM'] = addr['FULL_ADDRESS'].apply(normalize_address)

# Prepare geometry and deduplicate
addr = addr[['ADDR_NORM', 'GEOMETRY']].copy()
addr = addr.drop_duplicates()
addr = gpd.GeoDataFrame(addr, geometry='GEOMETRY', crs='EPSG:2264')

# === Merge and Geocode ===
geocoded = voters.merge(addr, on='ADDR_NORM', how='left')
geocoded = geocoded[geocoded['GEOMETRY'].notnull()]

# Convert to GeoDataFrame
geocoded_gdf = gpd.GeoDataFrame(geocoded, geometry='GEOMETRY', crs=addr.crs)

# === Save Result ===
print("Successfully geocoded:", len(geocoded_gdf))
geocoded_gdf.to_file(os.path.join(output_dir, "pitt_geocoded_voters.gpkg"), driver="GPKG", layer="pitt_geocoded_voters")


## Cell 4: Spatial Join to Parcels

In [ ]:
import os
import geopandas as gpd

# Path to parcel shapefile
parcel_path = str(BASE / "pitt-parcels-07-11-2025/nc_pitt_parcels_poly.shp")

# Read shapefile
parcels = gpd.read_file(parcel_path)
print("Loaded parcels:", len(parcels))

# Confirm geometry is properly set
print("Geometry column name:", parcels.geometry.name)

# Drop parcels without geometry
parcels = parcels.dropna(subset=['geometry'])

# Keep only relevant columns — don't rename 'geometry'
# Check if 'PARVAL' exists first
if 'PARVAL' in parcels.columns:
    parcels = parcels[['PARVAL', 'geometry']]
else:
    print("⚠ Column 'PARVAL' not found! Available columns:")
    print(parcels.columns)
    raise KeyError("Missing expected column: PARVAL")

# Ensure coordinate systems match before spatial join
parcels = parcels.to_crs(geocoded_gdf.crs)

# Spatial join: voter points inside parcel polygons
voters_on_parcels = gpd.sjoin(geocoded_gdf, parcels, how='inner', predicate='within')

# Drop duplicates
voters_on_parcels = voters_on_parcels.drop_duplicates(subset='NCID')

# Output result
print("Voters joined to parcels:", len(voters_on_parcels))
voters_on_parcels.to_file(
    os.path.join(output_dir, "pitt_voters_on_parcels.gpkg"),
    driver="GPKG",
    layer="pitt_voters_on_parcels"
)


### 📊 1. Average Property Value by Political Affiliation

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Plot
plt.figure(figsize=(8, 5))
sns.barplot(
    data=voters_on_parcels,
    x='PARTY_CD',
    y='PARVAL',
    estimator='mean',
    palette='Set2'
)
plt.ylabel('Average Property Value ($)')
plt.title('Avg Property Value by Political Affiliation (Pitt)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Save the figure
plt.savefig(os.path.join(output_dir, "pitt_avg_age_by_political_assoc.png"))
plt.show()


### 📊 2. Average Property Value by Sex of Voter

In [ ]:
import os
plt.figure(figsize=(8, 5))
sns.barplot(
    data=voters_on_parcels,
    x='GENDER_CODE',
    y='PARVAL',
    estimator='mean',
    palette='pastel'
)
plt.ylabel('Average Property Value ($)')
plt.title('Avg Property Value by Sex (Pitt)')
plt.xticks(rotation=0)
plt.tight_layout()

# Save the figure
plt.savefig(os.path.join(output_dir, "pitt_avg_property_value_by_sex.png"))
plt.show()


### 📊 3. Voter Count by Political Affiliation

In [ ]:
import os
plt.figure(figsize=(8, 5))
sns.countplot(
    data=voters_on_parcels,
    x='PARTY_CD',
    palette='Set2'
)
plt.ylabel('Voter Count')
plt.title('Voter Count by Political Affiliation (Pitt)')
plt.xticks(rotation=45)
plt.tight_layout()

# Save the figure
plt.savefig(os.path.join(output_dir, "pitt_avg_voter_count_political_affiliation.png"))
plt.show()


### 📊 4. Voter Count by Age Group

In [ ]:
import os
voters_on_parcels['AGE_AT_YEAR_END'] = pd.to_numeric(voters_on_parcels['AGE_AT_YEAR_END'], errors='coerce')
bins = [0, 17, 24, 34, 44, 54, 64, 74, 84, 150]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+']
voters_on_parcels['AGE_GROUP'] = pd.cut(voters_on_parcels['AGE_AT_YEAR_END'], bins=bins, labels=labels)

plt.figure(figsize=(10, 5))
sns.countplot(
    data=voters_on_parcels,
    x='AGE_GROUP',
    order=labels,
    palette='Blues'
)
plt.ylabel('Voter Count')
plt.title('Voter Count by Age Group (Pitt)')
plt.xticks(rotation=0)
plt.tight_layout()

# Save the figure
plt.savefig(os.path.join(output_dir, "pitt_avg_voter_count_by_age_group.png"))
plt.show()


### 📊 5. Voter Count by Race/Ethnicity

In [ ]:
import os
race_map = {
    'W': 'White', 'B': 'Black', 'U': 'Undesignated', 'O': 'Other',
    'A': 'Asian', 'M': 'Multi-racial', 'I': 'American Indian'
}
voters_on_parcels['RACE_LABEL'] = voters_on_parcels['RACE_CODE'].map(race_map).fillna('Unknown')

plt.figure(figsize=(10, 5))
sns.countplot(
    data=voters_on_parcels,
    x='RACE_LABEL',
    order=voters_on_parcels['RACE_LABEL'].value_counts().index,
    palette='coolwarm'
)
plt.ylabel('Voter Count')
plt.title('Voter Count by Race/Ethnicity (Pitt)')
plt.xticks(rotation=45)
plt.tight_layout()

# Save the figure
plt.savefig(os.path.join(output_dir, "pitt_avg_voter_count_by_race.png"))
plt.show()



### 📊 6. Average Age of Voters by Political Affiliation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Plot: Average Age of Voters by Party
plt.figure(figsize=(8, 5))
sns.barplot(
    data=voters_on_parcels,
    x='PARTY_CD',
    y='AGE_AT_YEAR_END',
    estimator=np.mean,
    palette='Set2'
)
plt.ylabel('Average Age')
plt.xlabel('Political Party')
plt.title('Average Age of Voters by Party (Pitt)')
plt.xticks(rotation=45)
plt.tight_layout()

# Save the figure
plt.savefig(os.path.join(output_dir, "pitt_avg_age_by_party.png"))
plt.show()

### ✅ Export Pitt County Voter Analysis to Excel

In [ ]:
import pandas as pd

# Fix AGE_GROUP column
bins = [0, 17, 24, 34, 44, 54, 64, 74, 84, 150]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+']
df = voters_on_parcels.copy()
df['AGE_GROUP'] = pd.cut(df['AGE_AT_YEAR_END'], bins=bins, labels=labels)

# Rename fields if needed for standardization
df = df.rename(columns={
    'PARTY_CD': 'party_cd',
    'GENDER_CODE': 'gender_code',
    'AGE_AT_YEAR_END': 'age_at_year_end'
})

# --- Analysis 1: Average Property Value by Political Affiliation
avg_val_party = df.groupby('party_cd')['PARVAL'].mean().round(2).sort_values(ascending=False).to_frame(name='Avg_Property_Value')

# --- Analysis 2: Average Property Value by Gender
avg_val_gender = df.groupby('gender_code')['PARVAL'].mean().round(2).sort_values(ascending=False).to_frame(name='Avg_Property_Value')

# --- Analysis 3: Voter Count by Political Affiliation
count_party = df['party_cd'].value_counts().sort_values(ascending=False).to_frame(name='Voter_Count')

# --- Analysis 4: Voter Count by Age Group
count_age_group = df['AGE_GROUP'].value_counts().sort_index().to_frame(name='Voter_Count')

# --- Analysis 5: Voter Count by Race/Ethnicity
count_race = df['RACE_LABEL'].value_counts().sort_values(ascending=False).to_frame(name='Voter_Count')

# --- Analysis 6: Average Age of Voters by Party
avg_age_party = df.groupby('party_cd')['age_at_year_end'].mean().round(2).sort_values(ascending=False).to_frame(name='Avg_Age')

# --- Write to Excel
with pd.ExcelWriter("pitt_summary_output.xlsx") as writer:
    avg_val_party.to_excel(writer, sheet_name='Avg_Property_by_Party')
    avg_val_gender.to_excel(writer, sheet_name='Avg_Property_by_Gender')
    count_party.to_excel(writer, sheet_name='Voter_Count_by_Party')
    count_age_group.to_excel(writer, sheet_name='Voter_Count_by_Age')
    count_race.to_excel(writer, sheet_name='Voter_Count_by_Race')
    avg_age_party.to_excel(writer, sheet_name='Avg_Age_by_Party')

print("Summary Excel file saved as 'pitt_summary_output.xlsx'")


In [ ]:
# Set output path
output_dir = "output"  # change if needed

# Save geocoded voter points
geocoded_gdf.to_file(f"{output_dir}/pitt_geocoded_voters.gpkg", driver="GPKG")
print("✅ Saved: pitt_geocoded_voters.gpkg")

# Save voter-parcel join with PARVAL
voters_on_parcels.to_file(f"{output_dir}/pitt_voters_on_parcels.gpkg", driver="GPKG")
print("✅ Saved: pitt_voters_on_parcels.gpkg")


In [ ]:
# Clean labels
df = voters_on_parcels.copy()
df['party_cd'] = df['PARTY_CD']
df['gender_label'] = df['GENDER_CODE'].map({'M': 'Male', 'F': 'Female'}).fillna('Unknown')
df['race_label'] = df['RACE_LABEL']
df['age_group'] = pd.cut(df['AGE_AT_YEAR_END'], bins=[0,18,25,35,45,55,65,75,85,200],
                         labels=["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85+"])


# --- Spatial Analyses ---
avg_val_party = df.groupby('party_cd')['PARVAL'].mean().round(2).sort_values(ascending=False)
avg_val_gender = df.groupby('gender_label')['PARVAL'].mean().round(2).sort_values(ascending=False)
count_party = df['party_cd'].value_counts().to_frame(name='Voter_Count')
count_age_group = df['age_group'].value_counts().sort_index().to_frame(name='Voter_Count')
count_race = df['race_label'].value_counts().to_frame(name='Voter_Count')
avg_age_party = df.groupby('party_cd')['AGE_AT_YEAR_END'].mean().round(2).sort_values(ascending=False)

# Display
print("Analysis 1: Avg Property Value by Political Affiliation\n", avg_val_party)
print("\nAnalysis 2: Avg Property Value by Gender\n", avg_val_gender)
print("\nAnalysis 3: Voter Count by Political Affiliation\n", count_party)
print("\nAnalysis 4: Voter Count by Age Group\n", count_age_group)
print("\nAnalysis 5: Voter Count by Race/Ethnicity\n", count_race)
print("\nAnalysis 6: Avg Age of Voters by Political Affiliation\n", avg_age_party)


In [ ]:
voters = pd.read_csv(
    str(BASE / "ncvoter74Pitt/ncvoter74.txt"),
    delimiter="\t",
    dtype=str,
    encoding="latin1",  # or try "ISO-8859-1"
    low_memory=False
)


In [ ]:
import geopandas as gpd

# Load geocoded and parcel-joined GeoPackage files
geocoded = gpd.read_file("output/pitt_geocoded_voters.gpkg")
joined = gpd.read_file("output/pitt_voters_on_parcels.gpkg")

# Total voter records from raw .txt file (already in variable: voters)
total_voters = len(voters)
geocoded_voters = len(geocoded)
parcel_joined_voters = len(joined)

# Percentages
percent_geocoded = (geocoded_voters / total_voters) * 100
percent_parcel_joined = (parcel_joined_voters / total_voters) * 100

# Print summary
print(f"Total voter records: {total_voters:,}")
print(f"Geocoded voters: {geocoded_voters:,} ({percent_geocoded:.1f}%)")
print(f"Voters joined to parcels: {parcel_joined_voters:,} ({percent_parcel_joined:.1f}%)")


In [ ]:
# Frequency count of party affiliations (from parcel-joined data)
party_counts = joined['PARTY_CD'].value_counts(dropna=False)
print("\nParty Breakdown:\n", party_counts)


# Wake

In [ ]:
import pandas as pd
import geopandas as gpd
import os
import re

# === Setup ===
base = str(BASE)
voter_path = os.path.join(base, "ncvoter92", "ncvoter92.txt")
addr_gdb = os.path.join(base, "WAKE-addresses-06-11-2025", "WAKE.gdb")
parcel_path = os.path.join(base, "wake-parcels-05-13-2025", "nc_wake_parcels_poly.shp")
output_dir = os.path.join(base, "output")
os.makedirs(output_dir, exist_ok=True)

# === Load Voter Data ===
voters = pd.read_csv(voter_path, sep="\t", dtype=str, encoding="latin1", low_memory=False)
print("Loaded:", len(voters))

voters.columns = voters.columns.str.upper()
voters = voters[voters['VOTER_STATUS_DESC'] == 'ACTIVE']
voters = voters.dropna(subset=['RES_STREET_ADDRESS', 'RES_CITY_DESC', 'STATE_CD', 'ZIP_CODE'])
voters['FULL_ADDRESS'] = (
    voters['RES_STREET_ADDRESS'].str.strip() + ', ' +
    voters['RES_CITY_DESC'].str.strip() + ', ' +
    voters['STATE_CD'].str.strip() + ' ' +
    voters['ZIP_CODE'].str.strip()
)
print("After cleaning:", len(voters))

# === Normalize Addresses ===
def normalize_address(s):
    if pd.isna(s): return ""
    s = s.lower()
    s = re.sub(r'#\s*\w+', '', s)
    s = re.sub(r'\bdr\b', 'drive', s)
    s = re.sub(r'\brd\b', 'road', s)
    s = re.sub(r'\bst\b', 'street', s)
    s = re.sub(r'\bave\b', 'avenue', s)
    s = re.sub(r'\bblvd\b', 'boulevard', s)
    s = re.sub(r'\bln\b', 'lane', s)
    s = re.sub(r'[^\w\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'nc\s*\d{5}$', '', s)
    return s

voters['ADDR_NORM'] = voters['FULL_ADDRESS'].apply(normalize_address)

# === Load Address Points ===
addr = gpd.read_file(addr_gdb, layer="WAKE")
addr.columns = addr.columns.str.upper()
addr['ADDR_NORM'] = addr['FULL_ADDRESS'].apply(normalize_address)
addr = addr[['ADDR_NORM', 'GEOMETRY']].drop_duplicates()
addr = gpd.GeoDataFrame(addr, geometry='GEOMETRY', crs='EPSG:2264')

# === Geocode Voters ===
geocoded = voters.merge(addr, on='ADDR_NORM', how='left')
geocoded = geocoded[geocoded['GEOMETRY'].notnull()]
geocoded_gdf = gpd.GeoDataFrame(geocoded, geometry='GEOMETRY', crs=addr.crs)
print("Successfully geocoded:", len(geocoded_gdf))

# === Save Geocoded Voters ===
geocoded_gdf.to_file(os.path.join(output_dir, "wake_geocoded_voters.gpkg"), driver="GPKG", layer="wake_geocoded_voters")

# === Load and Join with Parcels ===
if not os.path.exists(parcel_path):
    raise FileNotFoundError(f"Parcel shapefile not found: {parcel_path}")

parcels = gpd.read_file(parcel_path)
parcels = parcels.dropna(subset=['geometry'])
parcels = parcels.to_crs(geocoded_gdf.crs)

if 'PARVAL' not in parcels.columns:
    raise KeyError("Missing expected column: PARVAL")

parcels = parcels[['PARVAL', 'geometry']]

# === Spatial Join ===
voters_on_parcels = gpd.sjoin(geocoded_gdf, parcels, how='inner', predicate='within')
voters_on_parcels = voters_on_parcels.drop_duplicates(subset='NCID')
print("Voters joined to parcels:", len(voters_on_parcels))

# === Save Spatial Join ===
voters_on_parcels.to_file(
    os.path.join(output_dir, "wake_voters_on_parcels.gpkg"),
    driver="GPKG",
    layer="wake_voters_on_parcels"
)

# === Basic Summary Statistics ===
total_voters = len(voters)
geocoded_voters = len(geocoded_gdf)
percent_geocoded = (geocoded_voters / total_voters) * 100
print(f"Geocoded {geocoded_voters} of {total_voters} voters ({percent_geocoded:.2f}%)")

joined_voters = len(voters_on_parcels)
percent_joined = (joined_voters / geocoded_voters) * 100
print(f"Voters joined to parcels: {joined_voters} ({percent_joined:.2f}%)")


### 1. 🔍 Analyze Property Values by Party

In [ ]:
voters_on_parcels.groupby('PARTY_CD')['PARVAL'].describe()

### 2. 📊 Plot property value distribution by party

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(data=voters_on_parcels, x='PARTY_CD', y='PARVAL')
plt.title("Property Value Distribution by Party")
plt.ylabel("Parcel Value")
plt.xlabel("Party")
plt.yscale("log")  # Optional: if PARVAL has wide range
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### ✅ Step 3: Export Summary Table to CSV

In [ ]:
import os
# Group by political party and summarize property value
summary = voters_on_parcels.groupby('PARTY_CD')['PARVAL'].describe()

# Export to CSV
summary_path = os.path.join(output_dir, "wake_property_value_summary_by_party.csv")
summary.to_csv(summary_path)

print("✅ Summary table saved to:", summary_path)


### ✅ Step 4: Create Choropleth or Buffer Maps (Python Version)

#### A. Plotting a Choropleth (Property Value by Voter)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
voters_on_parcels.plot(column='PARVAL', cmap='viridis', legend=True, ax=ax, markersize=0.5)
plt.title("Parcel Value by Voter Location (Wake County)")
plt.axis('off')
plt.tight_layout()
plt.show()


#### B. Creating Buffers (e.g., 500 meters around voters)

In [ ]:
import geopandas as gpd
import os

# Set paths
output_dir = str(BASE / "output")
buffer_path = os.path.join(output_dir, "wake_voter_buffers_500ft.gpkg")

# Create a copy of the voter points
voters_buffer = voters_on_parcels.copy()

# If multiple geometry columns exist, drop the non-active ones first
if 'geometry' in voters_buffer and hasattr(voters_buffer, 'geometry'):
    # Drop all columns that are also GeoSeries but not the active one
    for col in voters_buffer.columns:
        if isinstance(voters_buffer[col], gpd.GeoSeries) and col != voters_buffer.geometry.name:
            voters_buffer = voters_buffer.drop(columns=[col])

# Create 500 ft buffers
voters_buffer['geometry'] = voters_buffer.geometry.buffer(500)

# Drop any previously stored geometry metadata
voters_buffer.set_geometry('geometry', inplace=True)

# Save to file
voters_buffer.to_file(buffer_path, driver="GPKG")
print("✅ Buffer map saved:", buffer_path)

### ✅ Step 5: Design 5 Custom Spatial Analyses

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import os

# Load voter+parcel GeoPackage if not already in memory
input_path = str(BASE / "output/wake_voters_on_parcels.gpkg")
voters = gpd.read_file(input_path)
output_dir = str(BASE / "output")


#### 1. 🏙️ Urban vs Rural Comparison

In [ ]:
import os
# Simple way: classify based on city name
urban_cities = ['RALEIGH', 'CARY', 'APEX', 'MORRISVILLE']
voters['AREA_TYPE'] = voters['RES_CITY_DESC'].apply(lambda x: 'Urban' if x in urban_cities else 'Rural')

# Group by Urban/Rural
urban_summary = voters.groupby('AREA_TYPE')['PARVAL'].describe()
urban_summary.to_csv(os.path.join(output_dir, "urban_rural_parval_summary.csv"))
print("✅ Saved: Urban vs Rural summary CSV")


#### 2. 🧑‍🎓 Young Voter Clusters (under age 25)

In [ ]:
import os
voters['AGE'] = pd.to_numeric(voters['AGE_AT_YEAR_END'], errors='coerce')
young_voters = voters[voters['AGE'] < 25]

# Save and plot
young_voters.to_file(os.path.join(output_dir, "young_voters.gpkg"), driver="GPKG")
print("✅ Saved: Young voters GeoPackage")

young_voters.plot(markersize=1, figsize=(8,8), color='blue')
plt.title("Young Voters (<25) in Wake County")
plt.axis('off')
plt.show()


#### 3. 🏫 Voters Near Wake Schools

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import os

# Paths
base = str(BASE)
voter_path = os.path.join(base, "output", "voters_near_schools.gpkg")
school_path = os.path.join(base, "Public_Schools.shp")
output_path = os.path.join(base, "output", "voters_near_schools_static.png")

# Load data
voters = gpd.read_file(voter_path)
schools = gpd.read_file(school_path)

# Reproject to Web Mercator for basemap
voters = voters.to_crs(epsg=3857)
schools = schools.to_crs(epsg=3857)

# Color-code by party
party_colors = {
    "DEM": "blue",
    "REP": "red",
    "UNA": "gray",
    "LIB": "purple"
}
voters["color"] = voters["PARTY_CD"].map(party_colors).fillna("black")

# Plot
fig, ax = plt.subplots(figsize=(12, 10))

# Plot voters and schools
voters.plot(ax=ax, color=voters["color"], markersize=5, alpha=0.6, label="Voters")
schools.buffer(500).plot(ax=ax, color='green', alpha=0.3, edgecolor='black', linewidth=0.2, label="School Buffer")

# Add reliable basemap (CartoDB Positron)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

# Title and styling
ax.set_title("Voters Within 500 ft of Schools – Wake County", fontsize=14)
ax.set_axis_off()
ax.legend(loc="lower left")

# Save output
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.close()

print("✅ Static map saved:", output_path)


#### 4. 🏘️ Political Party Density Map

In [ ]:
plt.figure(figsize=(10, 8))
voters.plot(column='PARTY_CD', markersize=1, cmap='Set1', legend=True)
plt.title("Voter Distribution by Party")
plt.axis('off')
plt.tight_layout()
plt.show()


#### 5. 🏞️ Distance to Nearest Park

In [ ]:
import geopandas as gpd
import pandas as pd
import os
from shapely.geometry import Point
import folium

# === Paths ===
base = str(BASE)
voter_path = os.path.join(base, "output", "wake_voters_on_parcels.gpkg")
parks_path = os.path.join(base, "Parks_and_Recreation_Sites.shp")
output_csv = os.path.join(base, "output", "voter_distance_to_park.csv")

# === Load voter and park data ===
voters = gpd.read_file(voter_path, layer="wake_voters_on_parcels")
parks = gpd.read_file(parks_path).to_crs(voters.crs)

# === Calculate minimum distance from each voter to nearest park ===
voters['DIST_TO_PARK'] = voters.geometry.apply(
    lambda pt: parks.distance(pt).min()
)

# === Save summary CSV ===
voters[['NCID', 'PARTY_CD', 'PARVAL', 'DIST_TO_PARK']].to_csv(output_csv, index=False)
print("✅ Saved CSV:", output_csv)

# === Optional: Preview (print a few lines) ===
print(voters[['NCID', 'PARTY_CD', 'DIST_TO_PARK']].head())

# === Bonus: Interactive Folium Map ===
# Create base map centered around Wake County
map_center = [voters.geometry.y.mean(), voters.geometry.x.mean()]
m = folium.Map(location=map_center, zoom_start=11)

# Add 100 voters only (to avoid performance issue)
sample = voters[['DIST_TO_PARK', 'geometry']].dropna().sample(100)

# Add each point to the map
for _, row in sample.iterrows():
    dist = round(row['DIST_TO_PARK'], 1)
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        popup=f"{dist} m",
        color="blue",
        fill=True,
        fill_opacity=0.5
    ).add_to(m)

# Save interactive map
map_path = os.path.join(base, "output", "wake_voter_park_map.html")
m.save(map_path)
print("🗺️ Interactive map saved at:", map_path)


In [ ]:
import geopandas as gpd
import pandas as pd

# ✅ Correct path to Wake voters joined to parcels
wake_path = str(BASE / "output/wake_voters_on_parcels.gpkg")

# ✅ Load spatially joined Wake voters
wake_voters = gpd.read_file(wake_path)

# ✅ Standardize column names to uppercase
wake_voters.columns = wake_voters.columns.str.upper()

# ✅ Check and filter relevant data
wake_voters = wake_voters.dropna(subset=['RACE_CODE', 'PARTY_CD', 'PARVAL'])

# ✅ Group and summarize
summary_wake = wake_voters.groupby(['RACE_CODE', 'PARTY_CD'])['PARVAL'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).reset_index()

# ✅ Save to CSV
summary_file_path = str(BASE / "output/summary_statistics_by_race_party_wake.csv")
summary_wake.to_csv(summary_file_path, index=False)

print(f"✅ Wake summary CSV saved to: {summary_file_path}")


## Pitt County

### ✅Pitt County Voter Geocoding and Parcel Join Code

In [ ]:
import os
import pandas as pd

# Path to the Pitt County voter file
pitt_voter_path = str(BASE / "ncvoter74Pitt/ncvoter74.txt")

# Step 1: Check if file exists
if not os.path.exists(pitt_voter_path):
    print(f"❌ File does NOT exist: {pitt_voter_path}")
else:
    print(f"✅ File exists: {pitt_voter_path}")

    # Step 2: Try to load a few lines to verify readability
    try:
        df = pd.read_csv(pitt_voter_path, sep="\t", encoding="latin1", dtype=str, nrows=5)
        print("📄 Successfully read sample data:")
        print(df.head())
        print("✅ Columns detected:", df.columns.tolist())
    except Exception as e:
        print(f"⚠️ Failed to read the file: {e}")


In [ ]:
import pandas as pd
import geopandas as gpd
import os
import re

# === File Paths ===
base = str(BASE)
voter_path = os.path.join(base, "ncvoter74Pitt", "ncvoter74.txt")
addr_gdb = os.path.join(base, "PITT-addresses-06-11-2025", "PITT.gdb")
output_dir = os.path.join(base, "output")
os.makedirs(output_dir, exist_ok=True)

# === Load Voter Data ===
print("🔍 Loading voter data...")
voters = pd.read_csv(voter_path, sep="\t", dtype=str, encoding="latin1", low_memory=False)
voters.columns = voters.columns.str.upper()
voters = voters[voters['COUNTY_DESC'] == 'PITT']
voters = voters[voters['VOTER_STATUS_DESC'] == 'ACTIVE']
voters = voters.dropna(subset=['RES_STREET_ADDRESS', 'RES_CITY_DESC', 'STATE_CD', 'ZIP_CODE'])

voters['FULL_ADDRESS'] = (
    voters['RES_STREET_ADDRESS'].str.strip() + ', ' +
    voters['RES_CITY_DESC'].str.strip() + ', ' +
    voters['STATE_CD'].str.strip() + ' ' +
    voters['ZIP_CODE'].str.strip()
)

print("✅ Pitt County ACTIVE voters loaded:", len(voters))

# === Normalize Addresses ===
def normalize_address(s):
    if pd.isna(s): return ""
    s = s.lower()
    s = re.sub(r'#\s*\w+', '', s)
    s = re.sub(r'\bdr\b', 'drive', s)
    s = re.sub(r'\brd\b', 'road', s)
    s = re.sub(r'\bst\b', 'street', s)
    s = re.sub(r'\bave\b', 'avenue', s)
    s = re.sub(r'\bblvd\b', 'boulevard', s)
    s = re.sub(r'\bln\b', 'lane', s)
    s = re.sub(r'[^\w\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'nc\s*\d{5}$', '', s)
    return s

voters['ADDR_NORM'] = voters['FULL_ADDRESS'].apply(normalize_address)

# === Load Address Points ===
print("📍 Loading PITT address points...")
addr = gpd.read_file(addr_gdb, layer="PITT")
addr.columns = addr.columns.str.upper()
addr['ADDR_NORM'] = addr['FULL_ADDRESS'].apply(normalize_address)
addr = addr[['ADDR_NORM', 'GEOMETRY']].drop_duplicates()
addr = gpd.GeoDataFrame(addr, geometry='GEOMETRY', crs='EPSG:2264')
print("✅ Address points loaded:", len(addr))

# === Geocode ===
geocoded = voters.merge(addr, on='ADDR_NORM', how='left')
geocoded = geocoded[geocoded['GEOMETRY'].notnull()]
geocoded_gdf = gpd.GeoDataFrame(geocoded, geometry='GEOMETRY', crs=addr.crs)
print("📌 Successfully geocoded:", len(geocoded_gdf))

# === Save Geocoded Voters ===
out_path = os.path.join(output_dir, "pitt_geocoded_voters.gpkg")
geocoded_gdf.to_file(out_path, driver="GPKG", layer="pitt_geocoded_voters")
print(f"💾 Geocoded voters saved to: {out_path}")


### ✅ Step 1: List Available Layers in The GeoPackage

In [ ]:
import fiona
from pathlib import Path

gpkg_path = str(BASE / "output/pitt_geocoded_voters.gpkg")

if Path(gpkg_path).exists():
    print("✅ File exists:", gpkg_path)
    print("📂 Available layers:")
    print(fiona.listlayers(gpkg_path))
else:
    print("❌ File does not exist.")


### 🔧 Step 2: Run This Code to Load the Geocoded Voters and Join with Parcels

In [ ]:
import geopandas as gpd
import os

# === Paths ===
base = str(BASE)
gpkg_path = os.path.join(base, "output", "pitt_geocoded_voters.gpkg")
parcel_path = os.path.join(base, "pitt-parcels-07-11-2025", "nc_pitt_parcels_poly.shp")
output_path = os.path.join(base, "output", "pitt_voters_on_parcels.gpkg")

# === Load Geocoded Voters ===
print("📥 Loading geocoded voters...")
geocoded_gdf = gpd.read_file(gpkg_path, layer="pitt_geocoded_voters")
print("✅ Loaded:", len(geocoded_gdf))

# === Load Parcels ===
print("📦 Loading parcels...")
parcels = gpd.read_file(parcel_path)
parcels = parcels.dropna(subset=['geometry'])
parcels = parcels.to_crs(geocoded_gdf.crs)

# Check if PARVAL exists
if 'PARVAL' not in parcels.columns:
    raise KeyError("❌ Missing expected column: PARVAL")

parcels = parcels[['PARVAL', 'geometry']]
print("✅ Parcels loaded:", len(parcels))

# === Spatial Join ===
print("🔗 Performing spatial join...")
voters_on_parcels = gpd.sjoin(geocoded_gdf, parcels, how='inner', predicate='within')
voters_on_parcels = voters_on_parcels.drop_duplicates(subset='NCID')
print("✅ Voters joined to parcels:", len(voters_on_parcels))

# === Save Result ===
voters_on_parcels.to_file(output_path, driver="GPKG", layer="pitt_voters_on_parcels")
print("💾 Saved to:", output_path)

### ✅ Spatial Analysis: Voter Preference vs. Property Value in Pitt County

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

# === Paths ===
gdf_path = str(BASE / "output/pitt_voters_on_parcels.gpkg")
gdf_layer = "pitt_voters_on_parcels"

print("🔍 Loading spatial join file...")
if not os.path.exists(gdf_path):
    print("❌ File not found:", gdf_path)
else:
    gdf = gpd.read_file(gdf_path, layer=gdf_layer)
    print("✅ File loaded:", len(gdf), "records")

    # === Check columns ===
    print("📑 Columns found:", list(gdf.columns))

    # === Preview party affiliation and parcel values ===
    if 'PARTY_CD' in gdf.columns and 'PARVAL' in gdf.columns:
        print("🔎 Sample data:")
        print(gdf[['PARTY_CD', 'PARVAL']].dropna().head())

        # === Clean data ===
        print("🧹 Cleaning data...")
        gdf = gdf[(gdf['PARVAL'].notna()) & (gdf['PARTY_CD'].notna())]
        gdf['PARVAL'] = pd.to_numeric(gdf['PARVAL'], errors='coerce')
        gdf = gdf[gdf['PARVAL'] > 0]

        print(f"✅ Cleaned data: {len(gdf)} valid records")

        # === Plot distribution ===
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=gdf, x='PARTY_CD', y='PARVAL')
        plt.yscale('log')
        plt.title("📊 Parcel Value by Voter Party (Pitt County)")
        plt.xlabel("Party")
        plt.ylabel("Parcel Value (log scale)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    else:
        print("❌ Required columns 'PARTY_CD' or 'PARVAL' not found.")

In [ ]:
import geopandas as gpd

# Path to your spatial join result
path = str(BASE / "output/pitt_voters_on_parcels.gpkg")

# Load and inspect
print("🔍 Loading spatial join file...")
gdf = gpd.read_file(path)
print(f"✅ File loaded: {len(gdf)} records")
print("📑 Columns found:")
print(gdf.columns.tolist())
print("🔎 Sample data:")
print(gdf[['PARTY_CD', 'PARVAL', 'geometry']].head())


### 🧠 Analyze and Visualize Property Values by Party Affiliation (Pitt County)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# === Load the spatial join file ===
print("🔍 Loading spatial join file...")
file_path = str(BASE / "output/pitt_voters_on_parcels.gpkg")

try:
    gdf = gpd.read_file(file_path)
    print(f"✅ File loaded: {len(gdf)} records")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    raise

# === Inspect and confirm required columns exist ===
required_columns = ['PARTY_CD', 'PARVAL']
missing = [col for col in required_columns if col not in gdf.columns]
if missing:
    print(f"❌ Missing required columns: {missing}")
else:
    print("📑 Columns found:")
    print(list(gdf.columns))
    print("🔎 Sample data:")
    print(gdf[['PARTY_CD', 'PARVAL', 'geometry']].head())

# === Clean and filter ===
print("🧹 Cleaning data...")
gdf = gdf[gdf['PARTY_CD'].notna() & gdf['PARVAL'].notna()]
gdf['PARVAL'] = pd.to_numeric(gdf['PARVAL'], errors='coerce')
gdf = gdf[gdf['PARVAL'] > 0]
print(f"✅ Cleaned data: {len(gdf)} valid records")

# === Aggregate statistics by party ===
print("📊 Calculating average parcel value by party...")
summary = gdf.groupby('PARTY_CD')['PARVAL'].agg(['count', 'mean', 'median']).sort_values(by='mean', ascending=False)
print(summary)

# === Plot boxplot of parcel value by party ===
print("📈 Plotting boxplot...")
plt.figure(figsize=(10, 6))
gdf.boxplot(column='PARVAL', by='PARTY_CD', grid=False)
plt.title("Parcel Value Distribution by Party Affiliation (Pitt County)")
plt.suptitle("")  # Remove auto title
plt.xlabel("Party Affiliation")
plt.ylabel("Parcel Value (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd

# Path to your GeoPackage file
gpkg_path = str(BASE / "output/pitt_voters_on_parcels.gpkg")

# 🔍 Load GeoPackage
print("🔍 Loading GeoPackage file...")
try:
    gdf = gpd.read_file(gpkg_path)
    print(f"✅ File loaded: {len(gdf)} records")
    
    # 📑 List columns
    print("📑 Columns found:")
    print(gdf.columns.tolist())

    # 🔎 Show sample rows
    print("🔎 Sample data:")
    print(gdf[['PARTY_CD', 'PARVAL', 'geometry']].head())
except Exception as e:
    print("❌ Failed to load file:", e)


### ✅ Task: Combined Spatial Analyses (Pitt County)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load spatial join file
print("🔍 Loading spatial join file...")
gdf = gpd.read_file(str(BASE / "output/pitt_voters_on_parcels.gpkg"))
print(f"✅ File loaded: {len(gdf)} records")

# Columns and sample
print("📑 Columns found:")
print(gdf.columns.tolist())
print("🔎 Sample data:")
print(gdf[['PARTY_CD', 'PARVAL', 'geometry']].head())

# Clean the data
print("🧹 Cleaning data...")
gdf = gdf[(gdf['PARVAL'].notna()) & (gdf['PARTY_CD'].notna())]
gdf['PARVAL'] = pd.to_numeric(gdf['PARVAL'], errors='coerce')
gdf = gdf[gdf['PARVAL'] > 0]
print(f"✅ Cleaned data: {len(gdf)} valid records")

# Summary stats
print("📊 Calculating average parcel value by party...")
summary = gdf.groupby("PARTY_CD")["PARVAL"].agg(['count', 'mean', 'median'])
print(summary)

# Boxplot
print("📈 Plotting boxplot...")
plt.figure(figsize=(10, 6))
sns.boxplot(x='PARTY_CD', y='PARVAL', data=gdf, showfliers=False)
plt.title("Parcel Value by Party Affiliation (Pitt County)")
plt.xlabel("Party")
plt.ylabel("Parcel Value ($)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Histogram with log scale
print("📊 Plotting log-transformed histogram...")
plt.figure(figsize=(12, 7))

# Filter and log-transform
gdf_log = gdf[gdf['PARVAL'] > 0].copy()
gdf_log['LOG_PARVAL'] = np.log10(gdf_log['PARVAL'])

# Plot per party
parties = gdf_log['PARTY_CD'].unique()
for party in parties:
    subset = gdf_log[gdf_log['PARTY_CD'] == party]
    plt.hist(subset['LOG_PARVAL'], bins=60, alpha=0.5, label=party)

plt.legend()
plt.title("Log-Scaled Distribution of Parcel Values by Party (Pitt County)")
plt.xlabel("Log10 of Parcel Value ($)")
plt.ylabel("Voter Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# Spatial scatter plot
print("🗺️ Plotting spatial scatter by party...")
fig, ax = plt.subplots(figsize=(10, 10))
colors = {'DEM': 'blue', 'REP': 'red', 'UNA': 'green', 'LIB': 'purple', 'GRE': 'orange'}
for party, color in colors.items():
    gdf[gdf['PARTY_CD'] == party].plot(ax=ax, markersize=1, label=party, color=color)
plt.title("Voter Locations by Party (Pitt County)")
plt.axis('off')
plt.legend()
plt.tight_layout()
plt.show()

### 🧪 Python Code – Political Clustering by Geography

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import os

# === Load Data ===
gdf = gpd.read_file(str(BASE / "output/pitt_voters_on_parcels.gpkg"))
gdf = gdf[(gdf['PARTY_CD'].notna()) & (gdf['PARVAL'].notna()) & (gdf['geometry'].notna())]
gdf = gdf.to_crs(epsg=2264)

# === Extract Coordinates ===
gdf['x'] = gdf.geometry.x
gdf['y'] = gdf.geometry.y

# === Define Party Colors ===
party_colors = {
    'DEM': 'blue',
    'REP': 'red',
    'UNA': 'green',
    'LIB': 'purple',
    'GRE': 'orange'
}

# === Plot KDEs ===
plt.figure(figsize=(14, 10))
for party, color in party_colors.items():
    if party in gdf['PARTY_CD'].unique():
        subset = gdf[gdf['PARTY_CD'] == party]
        sns.kdeplot(
            x=subset['x'],
            y=subset['y'],
            fill=True,
            thresh=0.05,
            alpha=0.3,
            color=color
        )

# === Manual Legend Handles ===
legend_handles = [
    Patch(facecolor=color, edgecolor='black', label=party, alpha=0.5)
    for party, color in party_colors.items()
    if party in gdf['PARTY_CD'].unique()
]

# === Final Touches ===
plt.title("Political Clustering by Geography (Pitt County)", fontsize=14)
plt.xlabel("Easting (meters)")
plt.ylabel("Northing (meters)")
plt.axis('equal')
plt.grid(alpha=0.3)
plt.legend(handles=legend_handles, title="Party", loc='upper right')
plt.tight_layout()

# === Save Figure ===
output_path = str(BASE / "output/pitt_party_clustering_kde.png")
plt.savefig(output_path, dpi=300)
print(f"✅ Legend fixed and figure saved to: {output_path}")

plt.show()

In [ ]:
import geopandas as gpd

# Load Pitt County voter-parcel data
gdf = gpd.read_file(str(BASE / "output/pitt_voters_on_parcels.gpkg"))

# Show column names and value counts for race and party
print("📑 Columns available:")
print(gdf.columns.tolist())

print("\n🔍 Unique races and counts:")
print(gdf['RACE_CODE'].value_counts(dropna=False))

print("\n🗳️ Unique party affiliations and counts:")
print(gdf['PARTY_CD'].value_counts(dropna=False))

print("\n💰 Sample parcel values:")
print(gdf['PARVAL'].dropna().sample(5, random_state=1))


### 🎯 Property Value Disparity by Race and Party

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load spatial join GeoPackage
gdf = gpd.read_file(str(BASE / "output/pitt_voters_on_parcels.gpkg"))

# Filter for valid values
gdf_clean = gdf.dropna(subset=['RACE_CODE', 'PARTY_CD', 'PARVAL'])
gdf_clean['PARVAL'] = gdf_clean['PARVAL'].astype(float)

# Cap PARVAL at 99th percentile to remove extreme outliers
upper_limit = gdf_clean['PARVAL'].quantile(0.99)
gdf_clean = gdf_clean[gdf_clean['PARVAL'] < upper_limit]

# Plot
plt.figure(figsize=(14, 7))
sns.boxplot(data=gdf_clean, x='RACE_CODE', y='PARVAL', hue='PARTY_CD')

plt.title("Property Value Disparity by Race and Party (Pitt County)", fontsize=16)
plt.xlabel("Race Code")
plt.ylabel("Parcel Value ($)")
plt.legend(title='Party', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()

# Save plot
output_path = str(BASE / "output/pitt_property_value_by_race_party.png")
plt.savefig(output_path, dpi=300)
plt.show()

print(f"✅ Plot saved to: {output_path}")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Filter valid entries
df = gdf[['RACE_CODE', 'PARTY_CD', 'PARVAL']].dropna()

# Step 2: Drop race groups with <30 voters
race_counts = df['RACE_CODE'].value_counts()
valid_races = race_counts[race_counts >= 30].index
df = df[df['RACE_CODE'].isin(valid_races)]

# Step 3: Apply IQR filtering per race
def iqr_filter(group):
    Q1 = group['PARVAL'].quantile(0.25)
    Q3 = group['PARVAL'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return group[(group['PARVAL'] >= lower) & (group['PARVAL'] <= upper)]

filtered = df.groupby('RACE_CODE', group_keys=False).apply(iqr_filter)

# Step 4: Plot
plt.figure(figsize=(16, 8))
sns.boxplot(
    data=filtered,
    x='RACE_CODE',
    y='PARVAL',
    hue='PARTY_CD',
    showfliers=True
)

plt.title("Property Value Disparity by Race and Party (Pitt County, Outliers Removed, N≥30)")
plt.xlabel("Race Code")
plt.ylabel("Parcel Value ($)")
plt.legend(title="Party", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

# Step 5: Save figure
output_path = str(BASE / "output/pitt_property_value_race_party_cleaned_final.png")
plt.savefig(output_path, dpi=300)
plt.show()


### 🧮 Summary Statistics by Race and Party

In [ ]:
import os
# Group by Race and Party and calculate summary statistics
summary_stats = (
    gdf.groupby(['RACE_CODE', 'PARTY_CD'])['PARVAL']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .reset_index()
    .sort_values(by=['RACE_CODE', 'PARTY_CD'])
)

# Round for readability
summary_stats[['mean', 'median', 'std', 'min', 'max']] = summary_stats[['mean', 'median', 'std', 'min', 'max']].round(2)

# Print the summary table
print(summary_stats)

# Optional: Export to CSV
output_path = os.path.join(output_dir, 'summary_statistics_by_race_party.csv')
summary_stats.to_csv(output_path, index=False)
print(f"\n✅ Summary statistics saved to: {output_path}")


In [ ]:
# ✅ Step 1: Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# ✅ Step 2: Load and Clean Data (Remove Outliers)
file_path = str(BASE / "output/summary_statistics_by_race_party.csv")
output_dir = str(BASE / "output")

# Load the CSV
df = pd.read_csv(file_path)

# Remove outliers using IQR for 'mean' and 'median'
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[column] >= lower) & (data[column] <= upper)]

df_clean = remove_outliers_iqr(df, 'mean')
df_clean = remove_outliers_iqr(df_clean, 'median')

# ✅ Step 3: Heatmap of Mean Parcel Value
pivot_heatmap = df_clean.pivot(index='RACE_CODE', columns='PARTY_CD', values='mean')
plt.figure(figsize=(12, 6))
sns.heatmap(pivot_heatmap, annot=True, fmt=".0f", cmap="coolwarm", linewidths=0.5, linecolor='gray', cbar_kws={'label': 'Mean Parcel Value'})
plt.title("Heatmap: Mean Parcel Value by Race and Party (Outliers Removed)")
plt.grid(True, which='both', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "heatmap_mean_parcel_value_cleaned.png"))
plt.show()

# ✅ Step 4: Bar Plot of Median Parcel Value
plt.figure(figsize=(12, 6))
sns.barplot(data=df_clean, x="RACE_CODE", y="median", hue="PARTY_CD")
plt.title("Bar Plot: Median Parcel Value by Race and Party (Outliers Removed)")
plt.ylabel("Median Parcel Value")
plt.xlabel("Race Code")
plt.grid(True, which='both', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "barplot_median_parcel_value_cleaned.png"))
plt.show()

# ✅ Step 5: Point Plot (Mean ± Std Dev, Log Scale)
plt.figure(figsize=(14, 6))
sns.pointplot(
    data=df_clean,
    x='RACE_CODE',
    y='mean',
    hue='PARTY_CD',
    dodge=0.5,
    join=False,
    capsize=0.2,
    errorbar='sd',
    markers='o'
)
plt.yscale('log')
plt.title('Mean Parcel Value by Race and Party with Std Dev (Log Scale)')
plt.ylabel('Mean Parcel Value (log scale)')
plt.xlabel('Race Code')
plt.grid(True, which='both', linestyle='--', alpha=0.3)
plt.legend(title='PARTY_CD')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'pointplot_mean_std_log.png'))
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# ✅ File paths
base_path = str(BASE / "output")
pitt_file = os.path.join(base_path, "summary_statistics_by_race_party_pitt.csv")
wake_file = os.path.join(base_path, "summary_statistics_by_race_party_wake.csv")

# ✅ Load and clean function
def load_and_clean(file):
    df = pd.read_csv(file)
    def remove_outliers_iqr(data, column):
        Q1 = data[column].quantile(0.25)
        Q3 = data[column].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        return data[(data[column] >= lower) & (data[column] <= upper)]
    df = remove_outliers_iqr(df, 'mean')
    df = remove_outliers_iqr(df, 'median')
    return df

# ✅ Load both counties
pitt_df = load_and_clean(pitt_file)
wake_df = load_and_clean(wake_file)

# ✅ Add county labels for combined plots
pitt_df['County'] = 'Pitt'
wake_df['County'] = 'Wake'
combined_df = pd.concat([pitt_df, wake_df], ignore_index=True)

# ---------- HEATMAP COMPARISON ----------
for county_name, df in [('Pitt', pitt_df), ('Wake', wake_df)]:
    pivot_heatmap = df.pivot(index='RACE_CODE', columns='PARTY_CD', values='mean')
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_heatmap, annot=True, fmt=".0f", cmap="coolwarm", linewidths=0.5, linecolor='gray', cbar_kws={'label': 'Mean Parcel Value'})
    plt.title(f"Heatmap: Mean Parcel Value by Race and Party ({county_name} County)")
    plt.grid(True, which='both', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(base_path, f"heatmap_{county_name.lower()}_mean.png"))
    plt.show()

# ---------- BAR PLOT COMPARISON ----------
plt.figure(figsize=(14, 6))
sns.barplot(data=combined_df, x='RACE_CODE', y='median', hue='County', ci=None)
plt.title("Bar Plot: Median Parcel Value by Race (Pitt vs Wake)")
plt.ylabel("Median Parcel Value")
plt.grid(True, which='both', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(base_path, "barplot_comparison_median.png"))
plt.show()

# ---------- POINT PLOT COMPARISON ----------
plt.figure(figsize=(14, 6))
sns.pointplot(
    data=combined_df,
    x='RACE_CODE',
    y='mean',
    hue='County',
    dodge=0.5,
    join=False,
    capsize=0.2,
    errorbar='sd',
    markers='o'
)
plt.yscale('log')
plt.title("Point Plot: Mean Parcel Value (Log Scale) with Std Dev (Pitt vs Wake)")
plt.ylabel("Mean Parcel Value (log scale)")
plt.grid(True, which='both', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(base_path, "pointplot_comparison_log_mean.png"))
plt.show()